Let's export the trained model to HuggingFace Hub in safetensors formats for compatibility with downstream inference engines. First, we'll define some variables.

In [1]:
model_name = "MewZoom-V1-2X"
checkpoint_path = "./checkpoints/2X-194.pt"
exports_path = "./exports"

Then, we'll load the base model checkpoint into memory from disk.

In [2]:
import torch

from src.mewzoom.model import MewZoom

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

upscaler = MewZoom(**checkpoint["upscaler_args"])

upscaler.model.add_qa_head(checkpoint["degradation_features"])
upscaler.model.add_weight_norms()

state_dict = checkpoint["upscaler"]

# Compensate for compiled state dict.
for key in list(state_dict.keys()):
    state_dict[key.replace("_orig_mod.", "")] = state_dict.pop(key)

upscaler.load_state_dict(state_dict)

upscaler.remove_parameterizations()
upscaler.model.remove_qa_head()

upscaler.eval()

print("Base checkpoint loaded successfully")

Base checkpoint loaded successfully


Now, let's export the model in HuggingFace format so that it can be used with the HuggingFace ecosystem.

In [3]:
from os import path

hf_path = path.join(exports_path, model_name)

upscaler.save_pretrained(hf_path)

print(f"Model saved to {hf_path}")

Model saved to ./exports/MewZoom-V1-2X


Next, we'll login to HuggingFaceHub and upload the model under our account.

In [4]:
from huggingface_hub import notebook_login

notebook_login()

upscaler.push_to_hub(model_name)

Uploading...:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/andrewdalpino/MewZoom-V1-2X/commit/452683cd7db68096d50693debd982db33128aa43', commit_message='Push model using huggingface_hub.', commit_description='', oid='452683cd7db68096d50693debd982db33128aa43', pr_url=None, repo_url=RepoUrl('https://huggingface.co/andrewdalpino/MewZoom-V1-2X', endpoint='https://huggingface.co', repo_type='model', repo_id='andrewdalpino/MewZoom-V1-2X'), pr_revision=None, pr_num=None)

Lastly, we'll export a model in ONNX format for use with the ONNX runtime.

In [5]:
from os import path

from torch.onnx import export as export_onnx

from torch.export.dynamic_shapes import Dim

from src.mewzoom.model import ONNXModel

onnx_path = path.join(exports_path, model_name, "model.onnx")

onnx_model = ONNXModel(upscaler)

x = torch.randn(1, 3, 128, 128)

example_input = (x,)

dynamic_shapes = {
    "x": {0: Dim.DYNAMIC, 1: Dim.STATIC, 2: Dim.DYNAMIC, 3: Dim.DYNAMIC},
}

onnx_graph = export_onnx(
    onnx_model,
    example_input,
    dynamic_shapes=dynamic_shapes,
    dynamo=True,
    input_names=["x"],
    output_names=["output"],
)

onnx_graph.save(onnx_path)

[torch.onnx] Obtain model graph for `ONNXModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ONNXModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 470 of general pattern rewrite rules.


With how haphazardly the ONNX support is implemented in PyTorch it's wise to do a quick sanity check on the newly exported ONNX model.

In [6]:
import onnxruntime

from numpy.testing import assert_allclose

pytorch_logits = upscaler.upscale(*example_input).detach().numpy()

session = onnxruntime.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

onnx_input = {"x": example_input[0].numpy()}

onnx_logits = session.run(None, onnx_input)

onnx_logits = onnx_logits[0]

assert_allclose(pytorch_logits, onnx_logits, rtol=1e-2, atol=1e-03)

print("Looks good!")

Looks good!
